# Data Cleaning and Preprocessing | Internship Milestone

**Goal:** Inspect a real dataset, document quality issues, handle missing values with column-specific reasoning, detect and assess outliers, identify duplicates, encode categorical variables, and compare the dataset before and after cleaning.

**Dataset:** Titanic passenger training dataset (891 records). It contains genuine missing values in `Age`, `Cabin`, and `Embarked`, and skewed numerical variables such as `Fare`.

**Important:** This notebook does not invent or inject data-quality problems. It reports the duplicates actually found and retains statistically unusual values when they are plausible observations rather than proven data-entry errors.

**How to use:** Open this file in Google Colab and run the cells from top to bottom. The dataset is downloaded from a public GitHub-hosted copy of the Titanic training CSV.

## 1. Import libraries and load the raw dataset

We keep an untouched copy (`raw_df`) so all before/after comparisons use the original data. The source is the commonly used Titanic training dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

DATA_URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

raw_df = pd.read_csv(DATA_URL)
df = raw_df.copy()

print("Dataset loaded successfully.")
print("Original shape:", raw_df.shape)
display(raw_df.head())

## 2. Initial data-quality assessment

First inspect data types, null counts, duplicate rows, unique values, and a sample. This assessment happens before cleaning so issues are not hidden by preprocessing.

In [ ]:
print("----- INFO -----")
raw_df.info()

print("\\n----- MISSING VALUES -----")
missing_before = raw_df.isna().sum().sort_values(ascending=False)
display(missing_before.to_frame("missing_count"))

print("\\n----- MISSING PERCENTAGE -----")
display((raw_df.isna().mean() * 100).round(2).to_frame("missing_percent"))

print("\\nExact duplicate rows:", raw_df.duplicated().sum())

print("\\n----- NUMERIC SUMMARY -----")
display(raw_df.describe().T)

print("\\n----- CATEGORICAL SUMMARY -----")
display(raw_df.describe(include="object").T)

print("\\n----- SAMPLE ROWS -----")
display(raw_df.sample(min(8, len(raw_df)), random_state=42))

## 3. Record baseline metrics

These values will be compared with the cleaned dataset at the end. We record both shape and missing-value counts before changing anything.

In [ ]:
baseline_shape = raw_df.shape
baseline_missing = raw_df.isna().sum()
baseline_numeric_summary = raw_df.select_dtypes(include=np.number).describe().T

print("Baseline rows:", baseline_shape[0])
print("Baseline columns:", baseline_shape[1])
display(baseline_missing.to_frame("missing_before"))
display(baseline_numeric_summary)

## 4. Handle duplicate rows

Exact duplicate rows can over-represent a record and distort counts or averages. We identify exact duplicates across all columns and remove them if present.

The notebook does not remove records merely because some fields match: two passengers can legitimately share a fare, ticket pattern, age, or cabin.

In [ ]:
duplicates_found = int(df.duplicated().sum())
print("Exact duplicate rows found:", duplicates_found)

if duplicates_found > 0:
    df = df.drop_duplicates().copy()
    print("Removed exact duplicate rows:", duplicates_found)
else:
    print("No exact duplicate rows were present; no rows removed.")

print("Shape after duplicate handling:", df.shape)

## 5. Handle missing values column by column

### Decisions and justification

| Column | Observed issue | Decision | Reason |
|---|---|---|---|
| `Age` | Missing passenger ages | Impute with median age within `Pclass` and `Sex` groups; fall back to overall median | Age is numeric and can be skewed. Group medians preserve broad age differences between passenger classes and sexes better than one global mean. |
| `Embarked` | A small number of missing embarkation ports | Impute with mode | It is a nominal category and the most frequent port is a reasonable choice for very few missing records. |
| `Cabin` | A very large proportion is missing | Drop the column | Most cabin values are absent, so simple imputation would create mostly invented information. |
| Other columns | Check actual null counts | Retain if no missing values | Do not alter columns without a demonstrated issue. |

We use `inplace=False`-style assignments and preserve the original raw dataframe for comparison.

In [ ]:
missing_before_imputation = df.isna().sum().sort_values(ascending=False)
display(missing_before_imputation.to_frame("missing_before_imputation"))

# Age: group median by passenger class and sex, then overall median as fallback.
if "Age" in df.columns and df["Age"].isna().any():
    age_group_medians = df.groupby(["Pclass", "Sex"], observed=False)["Age"].transform("median")
    df["Age"] = df["Age"].fillna(age_group_medians)
    df["Age"] = df["Age"].fillna(df["Age"].median())

# Embarked: nominal category; mode is appropriate for the very small number of nulls.
if "Embarked" in df.columns and df["Embarked"].isna().any():
    embarked_mode = df["Embarked"].mode(dropna=True)
    if not embarked_mode.empty:
        df["Embarked"] = df["Embarked"].fillna(embarked_mode.iloc[0])

# Cabin: drop because most values are missing and a useful imputation is not defensible.
if "Cabin" in df.columns:
    cabin_missing_pct = df["Cabin"].isna().mean() * 100
    print(f"Cabin missing: {cabin_missing_pct:.1f}%")
    if cabin_missing_pct > 50:
        df = df.drop(columns=["Cabin"])

print("Missing values after imputation/drop:")
display(df.isna().sum().to_frame("missing_after"))

## 6. Detect and assess numerical outliers using the IQR method

For each numeric variable (excluding the passenger identifier and target label), calculate:

- IQR = Q3 − Q1
- Lower fence = Q1 − 1.5 × IQR
- Upper fence = Q3 + 1.5 × IQR

**Decision policy:** An IQR flag means a value is unusual relative to the distribution; it does not prove that the value is wrong. We inspect the flagged values and retain plausible observations. In this dataset, high fares and larger family counts can be genuine passenger records. We do not delete or cap them without evidence of data-entry error.

`PassengerId` is an identifier, and `Survived` is the outcome label, so neither is treated as a continuous measurement for outlier removal.

In [ ]:
numeric_cols_for_outliers = [
    col for col in df.select_dtypes(include=np.number).columns
    if col not in ["PassengerId", "Survived"]
]

outlier_rows = []
outlier_masks = {}

for col in numeric_cols_for_outliers:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (df[col] < lower) | (df[col] > upper)
    outlier_masks[col] = mask

    outlier_rows.append({
        "column": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_fence": lower,
        "upper_fence": upper,
        "flagged_count": int(mask.sum()),
        "decision": "Retain; IQR flags are not proof of an error"
    })

outlier_report = pd.DataFrame(outlier_rows)
display(outlier_report)

### Visual inspection with box plots

Box plots help inspect the spread and IQR-flagged values. We retain plausible extremes rather than automatically deleting them.

In [ ]:
for col in numeric_cols_for_outliers:
    plt.figure(figsize=(8, 2.8))
    plt.boxplot(df[col].dropna(), vert=False)
    plt.title(f"Box plot: {col}")
    plt.xlabel(col)
    plt.show()

# Inspect some IQR-flagged records for the most skewed/common numeric field, Fare.
if "Fare" in outlier_masks:
    print("Sample IQR-flagged Fare observations (retained):")
    display(df.loc[outlier_masks["Fare"], ["PassengerId", "Pclass", "Fare"]].head(15))

## 7. Encode categorical variables

Encoding is selected according to the meaning of each variable:

- `Sex`: one-hot encoding because male/female are nominal categories with no natural ranking.
- `Embarked`: one-hot encoding because port names are nominal categories.
- `Pclass`: ordinal encoding because passenger classes have a meaningful order. We map first class to 1, second to 2, and third to 3. The numeric direction is documented: a larger code means a lower passenger class.
- `Survived`: retained as a 0/1 target label, not encoded as an input feature.

We drop `Name` and `Ticket` from this modelling-ready table because they are high-cardinality text/identifier fields and need separate feature engineering. `PassengerId` is also an identifier, not a predictive measurement.

In [ ]:
encoded_df = df.copy()

# Ordinal encoding for passenger class (ordered category).
if "Pclass" in encoded_df.columns:
    encoded_df["Pclass"] = encoded_df["Pclass"].map({1: 1, 2: 2, 3: 3}).astype("Int64")

# One-hot encoding for nominal categories.
nominal_cols = [c for c in ["Sex", "Embarked"] if c in encoded_df.columns]
encoded_df = pd.get_dummies(
    encoded_df,
    columns=nominal_cols,
    prefix=nominal_cols,
    dtype=int,
    drop_first=False
)

# Remove identifiers and high-cardinality raw text fields from the modelling-ready version.
columns_to_drop = [c for c in ["PassengerId", "Name", "Ticket"] if c in encoded_df.columns]
cleaned_df = encoded_df.drop(columns=columns_to_drop).copy()

print("Encoded dataset shape:", encoded_df.shape)
print("Final modelling-ready dataset shape:", cleaned_df.shape)
display(cleaned_df.head())
print("\\nFinal data types:")
display(cleaned_df.dtypes.to_frame("dtype"))

## 8. Validate the cleaned dataset

Confirm that missing values are handled, duplicate rows are absent, and categorical columns have been encoded. The target `Survived` remains available for a later supervised-learning task.

In [ ]:
print("Remaining missing values:", int(cleaned_df.isna().sum().sum()))
print("Remaining exact duplicate rows:", int(cleaned_df.duplicated().sum()))
print("Final shape:", cleaned_df.shape)

display(cleaned_df.isna().sum().to_frame("missing_after_cleaning"))
display(cleaned_df.head())

## 9. Before-and-after comparison

The raw dataset and final modelling-ready dataset have different column counts partly because categorical columns were expanded into indicator columns and identifier/text columns were removed. Therefore, we report both:
1. Raw dataset after quality cleaning but before encoding/dropping modelling identifiers (`df`).
2. Final modelling-ready dataset (`cleaned_df`).

The missing-value comparison below uses the raw columns that remain in `df`, so the effect of imputation and dropping `Cabin` is clear. Numeric summaries compare common numeric columns between the original and cleaned data.

In [ ]:
comparison = pd.DataFrame({
    "before_rows": [raw_df.shape[0]],
    "before_columns": [raw_df.shape[1]],
    "after_quality_cleaning_rows": [df.shape[0]],
    "after_quality_cleaning_columns": [df.shape[1]],
    "final_model_ready_rows": [cleaned_df.shape[0]],
    "final_model_ready_columns": [cleaned_df.shape[1]],
    "missing_cells_before": [int(raw_df.isna().sum().sum())],
    "missing_cells_after_quality_cleaning": [int(df.isna().sum().sum())],
    "missing_cells_final": [int(cleaned_df.isna().sum().sum())],
    "duplicates_before": [int(raw_df.duplicated().sum())],
    "duplicates_after": [int(cleaned_df.duplicated().sum())]
})
display(comparison.T.rename(columns={0: "value"}))

# Per-column missing counts for the original columns still present after quality cleaning.
missing_comparison = pd.DataFrame({
    "before": raw_df.isna().sum(),
    "after_quality_cleaning": df.reindex(columns=raw_df.columns).isna().sum()
})
display(missing_comparison)

# Compare numeric summary statistics for columns present in both datasets.
common_numeric = [
    c for c in raw_df.select_dtypes(include=np.number).columns
    if c in df.columns
]
summary_before = raw_df[common_numeric].describe().T.add_prefix("before_")
summary_after = df[common_numeric].describe().T.add_prefix("after_")
summary_comparison = summary_before.join(summary_after)
display(summary_comparison)

## 10. Save the cleaned dataset

Download a CSV copy of the cleaned, encoded dataset from Colab. This file can be used for later analysis or modelling.

**Note:** The output intentionally excludes `PassengerId`, `Name`, and `Ticket` as documented above. The `Survived` column remains the target label.

In [ ]:
output_file = "titanic_cleaned_encoded.csv"
cleaned_df.to_csv(output_file, index=False)
print(f"Saved: {output_file}")

try:
    from google.colab import files
    files.download(output_file)
except ImportError:
    print("Not running in Google Colab. The CSV has been saved in the current working directory.")

## 11. Conclusion

This workflow assessed the raw data before cleaning, identified and handled exact duplicates, applied justified missing-value strategies by column, detected numerical outliers using IQR and visual inspection, encoded nominal and ordinal categories appropriately, and compared the dataset before and after processing.

A key data-cleaning principle demonstrated here is that unusual values should be investigated rather than automatically removed. The IQR method identifies statistical extremes, but domain reasoning determines whether those values are errors. All decisions are documented so the cleaning process is transparent and reproducible.

## Submission checklist

- [ ] Run every cell from top to bottom in Google Colab.
- [ ] Check the printed missing-value, duplicate, and outlier reports.
- [ ] Review the before/after comparison and summary statistics.
- [ ] Ensure the notebook outputs are visible.
- [ ] Rename the notebook clearly, for example `Data_Cleaning_Milestone.ipynb`.
- [ ] Save the notebook to Google Drive.
- [ ] Upload the notebook to a public GitHub repository.
- [ ] Submit the public GitHub repository URL to Valentius Kryptix.

**Do not claim that duplicates were removed if the report shows zero duplicates.** Reporting that none were present is valid because the requirement is to identify and handle them honestly.